In [1]:
# single cell libraries
import scanpy as sc
import anndata as ad
import scrublet as scr
import leidenalg, igraph
import bbknn

# processing tools
import numpy as np
import pandas as pd
import scipy
import itertools

# plotting tools
import matplotlib.pyplot as plt

# file grab
from pathlib import Path
import pooch
import hashlib

/root/miniconda/envs/unified_env/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
/root/miniconda/envs/unified_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
sc.settings.verbosity = 2
sc.logging.print_versions()
results_file = "./write/_"

sc.settings.set_figure_params(dpi=80, frameon=False, figsize=(3, 3), facecolor="white")

In [3]:
# data download (zheng17 PBMC68k) 
SAMPLE_DATA = pooch.create( 
    path=pooch.os_cache("trajectory_practice"), 
    base_url="doi:10.6084/m9.figshare.14465568.v1/",
) 

SAMPLE_DATA.load_registry_from_doi()
fname = "pbmc68k.h5ad"

path = SAMPLE_DATA.fetch(fname)

# adata = sc.read_10x_h5(path) -> raw hdf5
adata = sc.read_h5ad(path)
print(adata)

## view file name
# print("Available files in registry:")
# print(SAMPLE_DATA.registry_files)

AnnData object with n_obs × n_vars = 65877 × 33939
    obs: 'celltype'
    var: 'Accession', 'Chromosome', 'End', 'Start', 'Strand'
    obsm: 'X_tsne'
    layers: 'spliced', 'unspliced'


In [25]:
# restore working adata
adata = sc.read_h5ad("cache/adata_pbmc68k.h5ad")
adata

AnnData object with n_obs × n_vars = 48256 × 139
    obs: 'celltype', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'doublet_score', 'predicted_doublet', 'leiden_r0.5', 'leiden_r0.8', 'leiden_r1.0', 'leiden_r1.2'
    var: 'Accession', 'Chromosome', 'End', 'Start', 'Strand', 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'leiden_r0.5', 'leiden_r0.5_colors', 'leiden_r0.8', 'leiden_r0.8_colors', 'leiden_r1.0', 'leiden_r1.0_colors', 'leiden_r1.2', 'leiden

In [4]:
#QC

# mitochondrial genes
adata.var["mt"] = adata.var_names.str.startswith("mt-")
# ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes
adata.var["hb"] = adata.var_names.str.contains("^HB[^(P)]")

In [5]:
sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True
)

sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
    save = "_qc_mt_counts",
)

In [6]:
print(adata.var_names[adata.var["mt"]].tolist())
print(adata.var["mt"].sum())

[]
0


In [7]:
sc.pl.scatter(
    adata, 
    "total_counts", "n_genes_by_counts",
    save = "_qc_counts",
)

In [7]:
# Filter minimum counts
sc.pp.filter_cells(adata, min_genes=100)
sc.pp.filter_genes(adata, min_cells=3) 

# Doublet detection
scrub = scr.Scrublet(adata.X)
doublet_scores, predicted_doublets = scrub.scrub_doublets()

# Remove doublets
adata.obs['doublet_score'] = doublet_scores
adata.obs['predicted_doublet'] = predicted_doublets
adata = adata[~adata.obs['predicted_doublet'], :]

filtered out 17621 cells that have less than 100 genes expressed
filtered out 21147 genes that are detected in less than 3 cells
Preprocessing...


/root/miniconda/envs/unified_env/lib/python3.10/site-packages/scrublet/helper_functions.py:254: RuntimeWarning: invalid value encountered in sqrt
  CV_input = np.sqrt(b);


Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.89
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 0.2%
Overall doublet rate:
	Expected   = 10.0%
	Estimated  = 0.0%
Elapsed time: 88.2 seconds


In [8]:
# Save count data
adata.layers["count"] = adata.X.copy()

# First normalization to handle dispersion-based HVG
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

# Set raw
X = adata.X.toarray() if scipy.sparse.issparse(adata.X) else adata.X
if np.any(np.isnan(X)) or np.any(np.isinf(X)):
    print("NaNs/Infs post-normalization. Check data.")
    raise
adata.raw = adata

normalizing counts per cell
    finished (0:00:00)


/tmp/ipykernel_887/1803662227.py:2: ImplicitModificationWarning: Setting element `.layers['count']` of view, initializing view as actual.
  adata.layers["count"] = adata.X.copy()


In [9]:
adata_raw = ad.AnnData(
    X=adata.raw.X,
    var=adata.raw.var,
    obs=adata.obs.copy()
)
adata_raw.write("cache/adata_raw.h5ad")

In [10]:
# HVG selection
sc.pp.highly_variable_genes(adata)

extracting highly variable genes
    finished (0:00:00)


In [11]:
sc.pp.pca(adata, svd_solver='arpack')
sc.pp.neighbors(adata, n_neighbors=20, n_pcs=40, metric="cosine")
adata

computing PCA
    with n_comps=50
    finished (0:00:00)
computing neighbors
    using 'X_pca' with n_pcs = 40
    finished (0:00:27)


AnnData object with n_obs × n_vars = 48256 × 12792
    obs: 'celltype', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'doublet_score', 'predicted_doublet'
    var: 'Accession', 'Chromosome', 'End', 'Start', 'Strand', 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'log1p', 'hvg', 'pca', 'neighbors'
    obsm: 'X_tsne', 'X_pca'
    varm: 'PCs'
    layers: 'spliced', 'unspliced', 'count'
    obsp: 'distances', 'connectivities'

In [12]:
sc.tl.umap(adata, min_dist=0.3, spread=1.0, init_pos="spectral")
sc.pl.umap(
    adata,
    size=4, alpha=0.8, legend_loc="on data", frameon=False,
    save="_1_pbmc68k_unlabeled"
)

computing UMAP
    finished (0:00:51)


In [13]:
RES_LIST = [0.5, 0.8, 1.0, 1.2]
fig, axes = plt.subplots(2, 2, figsize=(8,8))

for ax, res in zip(axes.flatten(), RES_LIST):
    key=f"leiden_r{res}"
    sc.tl.leiden(adata, flavor="igraph", resolution=res, key_added=key, random_state=0)
    sc.pl.embedding(adata, basis="umap", color=key, 
                    ax=ax, show=False, legend_loc="on data",
                    frameon=False, size=4, alpha=0.8,
                    title=f"res={res}",
                   )

plt.tight_layout()
plt.savefig('./figures/umap_2_pbmc68k_leiden.png')

running Leiden clustering
    finished (0:00:01)
running Leiden clustering
    finished (0:00:01)
running Leiden clustering
    finished (0:00:01)
running Leiden clustering
    finished (0:00:01)


In [14]:
# roughly annotating clusters via canonical markers
sc.pl.umap(
    adata,
    color=["CD3D","IL7R","CD8A","CD14","LYZ","CD19", "MS4A1", "PPBP"],
    cmap="viridis",
    vmin=0, vmax="p99",
    save="_3_canonicalMarkers3.png",
)


In [8]:
# annotation + marker gene extraction

sc.tl.rank_genes_groups(adata, groupby="leiden_r0.8", method="wilcoxon")
de = sc.get.rank_genes_groups_df(adata, group=None)  # 전체 합침

# 상수처럼 도는 것(리보솜/미토/히스토/스트레스) 제거
bad = ("^RPL|^RPS|^MT-|^HSP|^MALAT1|^FOS$|^JUN$|^JUNB$|^DUSP1$")
de = de[~de['names'].str.contains(bad, regex=True)]

# each cluster 상위 50~100개만 유지
de = de.groupby('group').head(80)

ranking genes
    finished (0:00:23)


/tmp/ipykernel_1262/2261401877.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  de = de.groupby('group').head(80)


In [15]:
marker_genes_broad = {
    "Mono" : ["CD14", "LYZ", "FCGR3A"],
    "NK" : ["NKG7", "FCGR3A"],
    "B" : ["BANK1", "CD79A"],
    "CD8 T" : ["CD3D", "CD8B"],
    "CD4 T" : ["CD3D", "IL7R"],
    "Platelet" : ["PPBP", "PF4"]
}

In [19]:
sc.pl.dotplot(
    adata, 
    marker_genes_broad, groupby="leiden_r1.0", standard_scale="var", 
    use_raw = True,
    dot_max = 0.5, dot_min = 0.05,
    save = "_3_r1.0.png"
)

In [22]:
marker_genes = {
    'T cells': ['IL7R', 'CD3D'],
    'B cells': ['CD79A'],
    'NK': ['GNLY', 'NKG7'],
    'Monocytes': ['LYZ', 'CD14'],
    'Platelets': ['PPBP']
}
sc.pl.dotplot(
    adata, 
    marker_genes, 
    groupby="leiden_r0.8", 
    use_raw=True, 
    dot_max=0.5, dot_min=0.05,
    save="_4.png"
)


In [23]:
############################################
# 작업중인 데이터 저장하기

adata.write("cache/adata_pbmc68k.h5ad")

In [9]:
"FCER1A" in adata.var_names

False

In [17]:
adata

AnnData object with n_obs × n_vars = 48256 × 12792
    obs: 'celltype', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'doublet_score', 'predicted_doublet', 'leiden_r0.5', 'leiden_r0.8', 'leiden_r1.0', 'leiden_r1.2'
    var: 'Accession', 'Chromosome', 'End', 'Start', 'Strand', 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'log1p', 'hvg', 'pca', 'neighbors', 'umap', 'leiden_r0.5', 'leiden_r0.5_colors', 'leiden_r0.8', 'leiden_r0.8_colors', 'leiden_r1.0', 'leid

In [ ]:
sc.pl.dotplot(
    adata, marker_genes, 
    groupby="leiden", 
    use_raw=True, 
    standard_scale="var",
    save = "_leiden",
)

In [ ]:
# testing optimal k-value, min_dist, resolutions

NB_LIST   = [15, 20, 30]
MD_LIST   = [0.2, 0.3, 0.4]
RES_LIST  = [0.5, 0.8, 1.0, 1.2, 1.5]

row_keys = list(itertools.product(NB_LIST, MD_LIST))
n_rows, n_cols = len(row_keys), len(RES_LIST)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.2*n_cols, 3.0*n_rows))
if n_rows == 1: axes = axes[None, :]
if n_cols == 1: axes = axes[:, None]

for r, (nb, md) in enumerate(row_keys):
    # neighbors+UMAP only once per (nb, md)
    sc.pp.neighbors(adata, n_neighbors=nb, n_pcs=40, metric="cosine")
    sc.tl.umap(adata, min_dist=md, spread=1.0, init_pos="spectral")
    # store layout so each column reuses it
    basis_key = f"umap_nb{nb}_md{md}"
    adata.obsm[f"X_{basis_key}"] = adata.obsm["X_umap"].copy()

    for c, res in enumerate(RES_LIST):
        key = f"leiden_r{res}_nb{nb}_md{md}"
        sc.tl.leiden(adata, key_added=key, resolution=res, random_state=0)

        ax = axes[r, c]
        sc.pl.embedding(
            adata,
            basis=basis_key, color=key,
            ax=ax, show=False, legend_loc=None,
            frameon=False, size=4, alpha=0.8
        )
        ax.set_title(f"nb={nb}, md={md}, res={res}", fontsize=9)

# tidy margins
plt.tight_layout()
plt.show()
plt.savefig('leidengrid')